# Kaggle API availability test

Quick smoke test for `stage2_event_expansion.py`'s external dependencies on Kaggle.
No model loading, no GPU. Just verifies:
1. Outbound HTTPS to Wikipedia
2. Outbound HTTPS to Tavily (with secret-loaded API key)

Make sure Kaggle notebook settings:
- **Internet: ON**
- **Secrets**: add `TAVILY_API_KEY`

In [ ]:
import os, re, sys
from urllib.parse import quote
import requests

from kaggle_secrets import UserSecretsClient
os.environ["TAVILY_API_KEY"] = UserSecretsClient().get_secret("TAVILY_API_KEY")
print("TAVILY_API_KEY loaded:", bool(os.environ.get("TAVILY_API_KEY")))

## 1. Wikipedia search + summary

Mirrors `_wiki_search` in `stage2_event_expansion.py`.

In [ ]:
_WIKI_SEARCH_URL = "https://en.wikipedia.org/w/api.php"
_WIKI_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/"
_UA = {"User-Agent": "finance-world-model/0.1 (research; contact via repo)"}

def wiki_search(query, k=3):
    resp = requests.get(
        _WIKI_SEARCH_URL,
        params={"action": "query", "list": "search", "srsearch": query,
                "srlimit": k, "format": "json"},
        headers=_UA, timeout=10,
    )
    resp.raise_for_status()
    out = []
    for r in resp.json().get("query", {}).get("search", []):
        title = r["title"]
        slug = quote(title.replace(" ", "_"))
        url = f"https://en.wikipedia.org/wiki/{slug}"
        try:
            s = requests.get(_WIKI_SUMMARY_URL + slug, headers=_UA, timeout=10)
            extract = s.json().get("extract", "") if s.status_code == 200 else ""
        except Exception:
            extract = ""
        if not extract:
            extract = re.sub(r"<[^>]+>", "", r.get("snippet", ""))
        out.append((title, extract, url))
    return out

hits = wiki_search("Federal Reserve December 2015 rate hike", k=3)
print(f"Wikipedia hits: {len(hits)}")
for i, (t, b, u) in enumerate(hits, 1):
    print(f"\n[{i}] {t}\n    {u}\n    {b[:200]}...")

## 2. Tavily search

Mirrors `_tavily_search` in `stage2_event_expansion.py`.

In [ ]:
_TAVILY_URL = "https://api.tavily.com/search"

def tavily_search(query, k=3):
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise RuntimeError("TAVILY_API_KEY missing from env")
    resp = requests.post(
        _TAVILY_URL,
        json={
            "api_key": api_key,
            "query": query,
            "max_results": k,
            "search_depth": "basic",
            "include_answer": False,
        },
        headers=_UA, timeout=15,
    )
    if resp.status_code >= 400:
        try:
            body = resp.json()
            msg = body.get("detail") or body.get("error") or body
        except ValueError:
            msg = resp.text[:300]
        raise RuntimeError(f"Tavily {resp.status_code}: {msg!r}")
    return [
        (r.get("title", ""), r.get("content", ""), r.get("url", ""))
        for r in resp.json().get("results", [])
    ]

hits = tavily_search("Federal Reserve December 2015 rate hike", k=3)
print(f"Tavily hits: {len(hits)}")
for i, (t, b, u) in enumerate(hits, 1):
    print(f"\n[{i}] {t}\n    {u}\n    {b[:200]}...")

## 3. Summary

If both cells above printed hits, Kaggle can reach the services `stage2_event_expansion.py` depends on for search.

In [ ]:
print("All API endpoints reachable from this Kaggle runtime.")